In [2]:
# proyecto_preparacion_datos_ecommerce.py
# Ejecutar: python proyecto_preparacion_datos_ecommerce.py

from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd


# =========================
# Configuración
# =========================
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "output"

CSV_PATH = DATA_DIR / "clientes_ecommerce.csv"
XLSX_PATH = DATA_DIR / "clientes_ecommerce.xlsx"

WEB_URL = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"  # solo para demostrar read_html()


def asegurar_carpetas() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Lección 1 - NumPy
# =========================
def leccion_1_numpy(seed: int = 42) -> Path:
    """
    Genera datos ficticios con NumPy y los guarda en .npy para usar en Pandas luego
    """
    rng = np.random.default_rng(seed)

    n_tx = 80
    tx_id = np.arange(1, n_tx + 1)
    cliente_id = rng.integers(1, 11, size=n_tx)  # IDs 1..10 (compatibles con tus archivos)

    # Montos con sesgo + outliers
    monto = (rng.lognormal(mean=7.1, sigma=0.35, size=n_tx) / 100).astype(float) * 1000
    out_idx = rng.choice(np.arange(n_tx), size=max(1, int(0.05 * n_tx)), replace=False)
    monto[out_idx] *= 8

    # Nulos artificiales
    null_idx = rng.choice(np.arange(n_tx), size=max(1, int(0.03 * n_tx)), replace=False)
    monto[null_idx] = np.nan

    categoria = rng.choice(["Electrónica", "Hogar", "Moda", "Deportes"], size=n_tx).astype(object)
    fecha_tx = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 120, size=n_tx), unit="D")

    # Operaciones NumPy
    print("Lección 1 (NumPy):")
    print(f"- Transacciones: {n_tx}")
    print(f"- Monto promedio (NaN ignorados): {np.nanmean(monto):.2f}")
    print(f"- Monto total (NaN ignorados): {np.nansum(monto):.2f}")

    datos = {
        "transacciones": {
            "TX_ID": tx_id,
            "ID": cliente_id,
            "Monto_TX": monto,
            "Categoria": categoria,
            "Fecha_TX": fecha_tx,
        }
    }

    out_npy = OUT_DIR / "transacciones_numpy.npy"
    np.save(out_npy, datos, allow_pickle=True)
    return out_npy


def cargar_transacciones_numpy(path_npy: Path) -> pd.DataFrame:
    datos = np.load(path_npy, allow_pickle=True).item()
    return pd.DataFrame(datos["transacciones"])


# =========================
# Lección 2 - Pandas
# =========================
def exploracion_pandas(df: pd.DataFrame, nombre: str) -> None:
    print(f"\nLección 2 (Pandas) - {nombre}")
    print(df.head(3))
    print(df.tail(3))
    print(df.describe(include="all"))


# =========================
# Lección 3 - Lectura CSV/Excel/Web
# =========================
def leer_clientes_csv_excel(csv_path: Path, xlsx_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_csv = pd.read_csv(csv_path)
    df_xlsx = pd.read_excel(xlsx_path)
    return df_csv, df_xlsx


def estandarizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza nombres (minúsculas, sin espacios) para merges más robustos.
    """
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    return out


def leer_tabla_web(url: str) -> pd.DataFrame:
    """
    Demostración de read_html() (si falla, se crea una tabla de referencia mínima)  .
    """
    try:
        tablas = pd.read_html(url)
        df_web = tablas[0].copy()
        df_web = df_web.head(8)  # reducir tamaño
    except Exception:
        df_web = pd.DataFrame({"fuente": ["fallback"], "valor": [1]})
    return df_web


def unificar_fuentes_clientes(df_csv: pd.DataFrame, df_xlsx: pd.DataFrame) -> pd.DataFrame:
    """
    concat + deduplicación por id (tus archivos comparten estructura)   .
    """
    df_all = pd.concat([df_csv, df_xlsx], ignore_index=True)
    df_all = estandarizar_columnas(df_all)

    # Deduplicar por 'id'
    if "id" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["id"], keep="first")

    return df_all


# =========================
# Lección 4 - Nulos y outliers
# =========================
def imputar_nulos_clientes(df: pd.DataFrame) -> pd.DataFrame:
    """
    - edad: mediana
    - ciudad/nombre: 'Desconocido' si faltara
    """
    out = df.copy()

    if "edad" in out.columns:
        out["edad"] = pd.to_numeric(out["edad"], errors="coerce")
        out["edad"] = out["edad"].fillna(out["edad"].median())

    for c in ["nombre", "ciudad"]:
        if c in out.columns:
            out[c] = out[c].fillna("Desconocido")

    for c in ["total_compras", "monto_total"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    return out


def tratar_outliers_iqr(df: pd.DataFrame, col: str, factor: float = 1.5) -> pd.DataFrame:
    """
    Trata outliers por IQR recortando (clip) para no perder filas  .
    """
    out = df.copy()
    x = pd.to_numeric(out[col], errors="coerce")

    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - factor * iqr, q3 + factor * iqr
    out[col] = x.clip(li, ls)
    return out


def tratar_outliers_zscore(df: pd.DataFrame, col: str, zmax: float = 3.0) -> pd.DataFrame:
    """
    Trata outliers por Z-score recortando (clip)  .
    """
    out = df.copy()
    x = pd.to_numeric(out[col], errors="coerce")

    mu = x.mean()
    sigma = x.std(ddof=0)
    if sigma == 0 or pd.isna(sigma):
        return out

    z = (x - mu) / sigma
    hi = x[z <= zmax].max()
    lo = x[z >= -zmax].min()
    out.loc[z > zmax, col] = hi
    out.loc[z < -zmax, col] = lo
    return out


# =========================
# Lección 5 - Wrangling
# =========================
def wrangling_clientes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Tipos
    out["id"] = pd.to_numeric(out["id"], errors="coerce").astype("Int64")
    out["nombre"] = out["nombre"].astype(str).str.strip().str.title()
    out["ciudad"] = out["ciudad"].astype(str).str.strip().str.title()

    # Columnas calculadas
    out["ticket_promedio"] = np.where(out["total_compras"] > 0, out["monto_total"] / out["total_compras"], 0)

    # Discretización de edad
    bins = [0, 25, 35, 50, 200]
    labels = ["<=25", "26-35", "36-50", "50+"]
    out["edad_bucket"] = pd.cut(out["edad"], bins=bins, labels=labels, include_lowest=True)

    # Normalización min-max del monto_total
    mn, mx = out["monto_total"].min(), out["monto_total"].max()
    out["monto_total_norm"] = (out["monto_total"] - mn) / (mx - mn) if mx != mn else 0.0

    return out


def wrangling_transacciones(df_tx: pd.DataFrame) -> pd.DataFrame:
    out = df_tx.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    out = out.drop_duplicates(subset=["tx_id"], keep="first")

    out["monto_tx"] = pd.to_numeric(out["monto_tx"], errors="coerce")
    out["fecha_tx"] = pd.to_datetime(out["fecha_tx"], errors="coerce")
    out["categoria"] = out["categoria"].astype(str).str.strip().str.title()

    out["mes_tx"] = out["fecha_tx"].dt.to_period("M").astype(str)
    out["monto_iva"] = out["monto_tx"].apply(lambda v: v * 1.19 if pd.notna(v) else v)

    return out


# =========================
# Lección 6 - Groupby / Pivot / Melt / Merge
# =========================
def consolidar_clientes_transacciones(df_clientes: pd.DataFrame, df_tx: pd.DataFrame) -> pd.DataFrame:
    """
    merge() entre transacciones y clientes por 'id' (cliente)  .
    """
    return df_tx.merge(df_clientes, on="id", how="left")


def agregados(df: pd.DataFrame) -> pd.DataFrame:
    """
    groupby() para métricas resumidas  .
    """
    req = {"ciudad", "categoria", "monto_tx", "id"}
    if not req.issubset(df.columns):
        return df.describe(include="all")

    out = (
        df.groupby(["ciudad", "categoria"], as_index=False)
          .agg(
              transacciones=("monto_tx", "size"),
              clientes_unicos=("id", "nunique"),
              monto_total_tx=("monto_tx", "sum"),
              monto_promedio_tx=("monto_tx", "mean"),
          )
          .sort_values(["ciudad", "monto_total_tx"], ascending=[True, False])
    )
    return out


def pivoteo_y_melt(df_agg: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    pivot() y melt()  .
    """
    pivot = df_agg.pivot(index="ciudad", columns="categoria", values="monto_total_tx").fillna(0)

    melt = df_agg.melt(
        id_vars=["ciudad", "categoria"],
        value_vars=["transacciones", "clientes_unicos", "monto_total_tx", "monto_promedio_tx"],
        var_name="metrica",
        value_name="valor",
    )
    return pivot, melt


def exportar(df_clientes: pd.DataFrame, df_tx: pd.DataFrame, df_final: pd.DataFrame,
             df_agg: pd.DataFrame, df_pivot: pd.DataFrame, df_melt: pd.DataFrame,
             df_web: pd.DataFrame) -> None:
    """
    Exporta dataset final a CSV y Excel + reporte con hojas  .
    """
    out_csv = OUT_DIR / "dataset_final.csv"
    out_xlsx = OUT_DIR / "dataset_final.xlsx"
    out_reporte = OUT_DIR / "reporte_completo.xlsx"

    df_final.to_csv(out_csv, index=False)
    df_final.to_excel(out_xlsx, index=False)

    with pd.ExcelWriter(out_reporte) as writer:
        df_clientes.to_excel(writer, sheet_name="clientes_limpios", index=False)
        df_tx.to_excel(writer, sheet_name="transacciones_limpias", index=False)
        df_final.to_excel(writer, sheet_name="dataset_final", index=False)
        df_agg.to_excel(writer, sheet_name="agregados", index=False)
        df_pivot.to_excel(writer, sheet_name="pivot_monto_total")
        df_melt.to_excel(writer, sheet_name="melt_metricas", index=False)
        df_web.to_excel(writer, sheet_name="tabla_web", index=False)

    print("\nArchivos exportados:")
    print(f"- {out_csv}")
    print(f"- {out_xlsx}")
    print(f"- {out_reporte}")


# =========================
# MAIN
# =========================
def main() -> None:
    asegurar_carpetas()

    # Lección 1: NumPy -> .npy
    path_npy = leccion_1_numpy(seed=42)

    # Lección 2: Pandas (cargar .npy)
    df_tx = cargar_transacciones_numpy(path_npy)
    exploracion_pandas(df_tx, "Transacciones (desde NumPy)")

    # Lección 3: leer CSV + Excel + web
    if not CSV_PATH.exists() or not XLSX_PATH.exists():
        raise FileNotFoundError(
            "Faltan archivos en ./data/. Copia clientes_ecommerce.csv y clientes_ecommerce.xlsx ahí."
        )

    df_csv, df_xlsx = leer_clientes_csv_excel(CSV_PATH, XLSX_PATH)
    exploracion_pandas(df_csv, "Clientes (CSV)")
    exploracion_pandas(df_xlsx, "Clientes (Excel)")

    df_web = leer_tabla_web(WEB_URL)

    df_clientes = unificar_fuentes_clientes(df_csv, df_xlsx)

    # Lección 4: nulos + outliers
    df_clientes = imputar_nulos_clientes(df_clientes)
    for col in ["monto_total", "total_compras", "edad"]:
        if col in df_clientes.columns:
            df_clientes = tratar_outliers_iqr(df_clientes, col=col, factor=1.5)
            df_clientes = tratar_outliers_zscore(df_clientes, col=col, zmax=3.0)

    # También limpiar transacciones (nulos/outliers en monto_tx)
    df_tx = wrangling_transacciones(df_tx)
    df_tx["monto_tx"] = df_tx["monto_tx"].fillna(df_tx["monto_tx"].median())
    df_tx = tratar_outliers_iqr(df_tx, col="monto_tx", factor=1.5)
    df_tx = tratar_outliers_zscore(df_tx, col="monto_tx", zmax=3.0)

    # Lección 5: wrangling avanzado
    df_clientes = wrangling_clientes(df_clientes)

    # Lección 6: merge + groupby + pivot + melt
    df_final = consolidar_clientes_transacciones(df_clientes, df_tx)
    df_agg = agregados(df_final)
    df_pivot, df_melt = pivoteo_y_melt(df_agg)

    # Exportación final
    exportar(df_clientes, df_tx, df_final, df_agg, df_pivot, df_melt, df_web)


if __name__ == "__main__":
    main()


Lección 1 (NumPy):
- Transacciones: 80
- Monto promedio (NaN ignorados): 16969.36
- Monto total (NaN ignorados): 1323610.43

Lección 2 (Pandas) - Transacciones (desde NumPy)
   TX_ID  ID      Monto_TX    Categoria   Fecha_TX
0      1   1  15720.550004         Moda 2025-02-25
1      2   8  14657.231156  Electrónica 2025-04-07
2      3   7   9601.327945         Moda 2025-04-26
    TX_ID  ID      Monto_TX Categoria   Fecha_TX
77     78   3  12212.720408     Hogar 2025-01-28
78     79   6  21230.788715     Hogar 2025-02-03
79     80   7  11145.717511  Deportes 2025-01-16
          TX_ID         ID       Monto_TX Categoria             Fecha_TX
count   80.0000  80.000000      78.000000        80                   80
unique      NaN        NaN            NaN         4                  NaN
top         NaN        NaN            NaN     Hogar                  NaN
freq        NaN        NaN            NaN        23                  NaN
mean    40.5000   5.650000   16969.364442       NaN  2025-02-